# 🧠 Notebook 01 — Data Extraction

**Goal:** Connect to the NeuroVault public API, extract brain imaging study data, and save it as CSV files for later analysis.

**What you will learn:**
- How to make HTTP requests to a REST API with `requests`
- How to parse JSON responses
- How to handle API pagination
- How to save data as CSV with `pandas`

**API base URL:** `https://neurovault.org/api/`  
**Authentication:** None required ✅

---
## 1. Import libraries

In [ ]:
import requests       # To make HTTP requests to the API
import pandas as pd   # To organize and save data
import time           # To pause between requests (good practice)
import os             # To create folders

print('Libraries imported successfully ✅')

---
## 2. Create the folder structure

We create the `data/raw/` folder where we will save the CSV files.

In [ ]:
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Folders created ✅')
print('  ../data/raw/       → original data from the API')
print('  ../data/processed/ → cleaned data (next notebook)')

---
## 3. Test the API connection

Before extracting everything, we test that the API is responding correctly.  
A status code of `200` means everything is OK.

In [ ]:
BASE_URL = 'https://neurovault.org/api'

response = requests.get(f'{BASE_URL}/collections/?limit=1')

print(f'Status code: {response.status_code}')

if response.status_code == 200:
    print('API connection successful ✅')
    data = response.json()
    print(f'Total collections available: {data["count"]}')
else:
    print('Connection failed ❌ — check your internet connection')

---
## 4. Explore one record

Before extracting everything, let's see what a single collection looks like — what fields it has and what data it contains.

In [ ]:
response = requests.get(f'{BASE_URL}/collections/?limit=1')
sample = response.json()['results'][0]

print('Fields available in a collection:')
print('-' * 40)
for key, value in sample.items():
    print(f'  {key}: {str(value)[:80]}')

---
## 5. Extract all collections with pagination

The API returns results in pages of 100 records at a time.  
We use a `while` loop to go through all pages until `next` is `None`.

> ⏱️ This may take a few minutes depending on how many records exist.

In [ ]:
def fetch_all_collections(limit=100):
    """
    Fetches all collections from the NeuroVault API.
    Handles pagination automatically.
    Returns a list of dictionaries.
    """
    all_records = []
    url = f'{BASE_URL}/collections/?limit={limit}'
    page = 1

    while url:
        print(f'  Fetching page {page}... ', end='')
        response = requests.get(url)

        if response.status_code != 200:
            print(f'Error on page {page}: {response.status_code}')
            break

        data = response.json()
        all_records.extend(data['results'])
        print(f'{len(data["results"])} records fetched (total so far: {len(all_records)})')

        url = data.get('next')   # Next page URL — None if last page
        page += 1
        time.sleep(0.5)          # Small pause to be respectful to the API

    return all_records


print('Starting extraction of collections...')
print('-' * 50)
collections = fetch_all_collections()
print('-' * 50)
print(f'\nExtraction complete ✅ — {len(collections)} collections fetched')

---
## 6. Convert to DataFrame and preview

In [ ]:
df_collections = pd.DataFrame(collections)

print(f'Shape: {df_collections.shape[0]} rows × {df_collections.shape[1]} columns')
print(f'\nColumns: {list(df_collections.columns)}')
df_collections.head(3)

---
## 7. Extract images (brain maps)

Now we extract the brain map images linked to those collections.

In [ ]:
def fetch_all_images(limit=100, max_pages=20):
    """
    Fetches brain map images from the NeuroVault API.
    max_pages limits the extraction to avoid very long runtimes.
    """
    all_records = []
    url = f'{BASE_URL}/images/?limit={limit}'
    page = 1

    while url and page <= max_pages:
        print(f'  Fetching page {page}/{max_pages}... ', end='')
        response = requests.get(url)

        if response.status_code != 200:
            print(f'Error: {response.status_code}')
            break

        data = response.json()
        all_records.extend(data['results'])
        print(f'{len(data["results"])} records (total: {len(all_records)})')

        url = data.get('next')
        page += 1
        time.sleep(0.5)

    return all_records


print('Starting extraction of images...')
print('-' * 50)
images = fetch_all_images(max_pages=20)
print('-' * 50)
print(f'\nExtraction complete ✅ — {len(images)} images fetched')

---
## 8. Save both datasets as CSV

In [ ]:
df_images = pd.DataFrame(images)

# Save to CSV
df_collections.to_csv('../data/raw/collections.csv', index=False)
df_images.to_csv('../data/raw/images.csv', index=False)

print('Files saved ✅')
print(f'  ../data/raw/collections.csv  → {df_collections.shape[0]} rows, {df_collections.shape[1]} columns')
print(f'  ../data/raw/images.csv       → {df_images.shape[0]} rows, {df_images.shape[1]} columns')

---
## 9. Quick summary

A first look at the data before moving to the cleaning notebook.

In [ ]:
print('=' * 50)
print('EXTRACTION SUMMARY')
print('=' * 50)
print(f'Collections extracted : {len(df_collections)}')
print(f'Images extracted      : {len(df_images)}')
print(f'Collection columns    : {df_collections.shape[1]}')
print(f'Image columns         : {df_images.shape[1]}')
print()
print('Null values in collections:')
print(df_collections.isnull().sum()[df_collections.isnull().sum() > 0])
print()
print('Next step → 02_cleaning.ipynb 🚀')

---
## ✅ What we accomplished

- Connected to the NeuroVault REST API without any authentication
- Handled pagination to extract all available records
- Saved two raw datasets: `collections.csv` and `images.csv`

**Next notebook:** `02_cleaning.ipynb` — we will clean nulls, fix data types, and prepare the data for analysis.